In [3]:
#setup
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mtb_drug_targets/'
!apt-get install -y ncbi-blast+ -q
!pip install biopython -q
!cp /content/drive/MyDrive/mtb_drug_targets/data/human_blast_db* /content/
print("Setup complete!")

Mounted at /content/drive
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  ncbi-data
The following NEW packages will be installed:
  ncbi-blast+ ncbi-data
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 15.8 MB of archives.
After this operation, 71.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-data all 6.1.20170106+dfsg1-9 [3,519 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-blast+ amd64 2.12.0+ds-3build1 [12.3 MB]
Fetched 15.8 MB in 0s (40.6 MB/s)
Selecting previously unselected package ncbi-data.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_a

In [4]:
#Load MTB data
import pandas as pd
df=pd.read_csv(BASE + 'results/final_candidates.csv')
print(df.columns.tolist())
print(df.shape)
print(df.head())

['query', 'subject', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
(2162, 12)
                      query                subject  pident  length  mismatch  \
0  sp|A0A089QRB9|MSL3_MYCTU   sp|P51659|DHB4_HUMAN  25.339     221       133   
1      sp|E2FZM4|SOCA_MYCTU  sp|Q9UQB3|CTND2_HUMAN  43.478      23        13   
2      sp|I6WXK4|PTPB_MYCTU   sp|P29350|PTN6_HUMAN  32.857      70        35   
3      sp|I6WZK7|MMCO_MYCTU  sp|Q6MZM0|HPHL1_HUMAN  35.000      60        33   
4      sp|I6X235|ADPP_MYCTU  sp|Q32P44|EMAL3_HUMAN  33.846      65        27   

   gapopen  qstart  qend  sstart  send  evalue  bitscore  
0        6    1742  1951       6   205   0.041      39.3  
1        0      18    40     923   945   0.870      26.9  
2        4     144   208     438   500   0.210      33.1  
3        1     437   496     304   357   0.002      41.2  
4        2       2    50     500   564   0.430      31.6  


In [5]:
#Load top 20 drug target candidates
df_top20 = df.sort_values("evalue", ascending=False).head(20)
print(df_top20[['query', 'evalue']])

                       query  evalue
15     sp|I6Y276|Y2993_MYCTU    1.00
753     sp|P9WKD7|RPIB_MYCTU    1.00
1558  tr|O06178|O06178_MYCTU    1.00
1320  tr|I6X7D4|I6X7D4_MYCTU    1.00
193    sp|P71592|WHB5A_MYCTU    1.00
1940  tr|P71813|P71813_MYCTU    1.00
1951  tr|P71898|P71898_MYCTU    0.99
1303  tr|I6WZ71|I6WZ71_MYCTU    0.99
1117    sp|P9WPK9|GCS2_MYCTU    0.99
214     sp|P95200|NDHA_MYCTU    0.99
886     sp|P9WM71|Y090_MYCTU    0.99
2094  tr|Q11064|Q11064_MYCTU    0.99
2014  tr|P95218|P95218_MYCTU    0.98
629    sp|P9WJ55|VAPB9_MYCTU    0.98
119    sp|O53281|Y3034_MYCTU    0.98
365     sp|P9WGI5|SIGB_MYCTU    0.98
1464  tr|I6YGT7|I6YGT7_MYCTU    0.98
2040  tr|P96238|P96238_MYCTU    0.98
1103     sp|P9WPG9|CDH_MYCTU    0.98
1462  tr|I6YGH7|I6YGH7_MYCTU    0.97


In [6]:
#saving the top 20 candidates to drive
df_top20.to_csv(BASE + 'results/top20_candidates.csv', index=False)
print("cleaner version saved!")

cleaner version saved!


In [7]:
#Merge with proteome to get sequences
df_proteome = pd.read_csv(BASE + 'results/proteome_final.csv')
df_top20_seq = df_top20.merge(
    df_proteome,
    left_on='query',
    right_on='Protein_ID'
)
print(df_top20_seq[['query', 'protein_name', 'evalue']].head())

                    query                            protein_name  evalue
0   sp|I6Y276|Y2993_MYCTU                         Protein Rv2993c     1.0
1    sp|P9WKD7|RPIB_MYCTU          Ribose-5-phosphate isomerase B     1.0
2  tr|O06178|O06178_MYCTU  Thioesterase domain-containing protein     1.0
3  tr|I6X7D4|I6X7D4_MYCTU                       Conserved protein     1.0
4   sp|P71592|WHB5A_MYCTU         Transcriptional regulator WhiB5     1.0


In [8]:
#Download Alphafold strucures for Top 20 candidates
import requests
import json
import os

def download_alphafold_structure(uniprot_id):
    clean_id = uniprot_id.split('|')[1]
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{clean_id}"
    api_response = requests.get(api_url, timeout=30)
    if api_response.status_code != 200:
        print(f"{clean_id} — not found in AlphaFold")
        return None
    data = json.loads(api_response.text)
    pdb_url = data[0]['pdbUrl']
    pdb_response = requests.get(pdb_url, timeout=30)
    if pdb_response.status_code == 200:
        pdb_path = BASE + f'results/structures/{clean_id}.pdb'
        with open(pdb_path, 'w') as f:
            f.write(pdb_response.text)
        print(f"{clean_id} — downloaded!")
        return pdb_path
    else:
        print(f"{clean_id} — download failed")
        return None
print("Function ready!")

Function ready!


In [9]:
#Download PDB Files for All 20 Candidates
pdb_files = []
for _, row in df_top20_seq.iterrows():
    pdb_path = download_alphafold_structure(row['query'])
    if pdb_path:
        pdb_files.append(pdb_path)
print(f"\nStructures downloaded: {len(pdb_files)}/{len(df_top20_seq)}")

I6Y276 — downloaded!
P9WKD7 — downloaded!
O06178 — downloaded!
I6X7D4 — downloaded!
P71592 — downloaded!
P71813 — downloaded!
P71898 — downloaded!
I6WZ71 — downloaded!
P9WPK9 — downloaded!
P95200 — downloaded!
P9WM71 — downloaded!
Q11064 — downloaded!
P95218 — downloaded!
P9WJ55 — downloaded!
O53281 — downloaded!
P9WGI5 — downloaded!
I6YGT7 — downloaded!
P96238 — downloaded!
P9WPG9 — downloaded!
I6YGH7 — downloaded!

Structures downloaded: 20/20


In [10]:
#Test visualization
!pip install py3Dmol -q
import py3Dmol
with open(BASE + 'results/structures/P9WKD7.pdb', 'r') as f:
    pdb_data = f.read()
view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [17]:
#visualizing the structures for all the 20 structures
for pdb_file in pdb_files:
    protein_id = pdb_file.split('/')[-1].replace('.pdb', '')
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()
    print(f"\nProtein: {protein_id}")
    view = py3Dmol.view(width=400, height=300)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({}, {'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()


Protein: I6Y276


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Protein: P9WKD7


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Protein: O06178


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Protein: P9WPG9


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Protein: I6YGH7


3Dmol.js failed to load for some reason. Please check your browser console for error messages.